# 📘 **Colab Notebook: Context Evaluation Using Llumo**

---

### 📝 **Notebook Overview**

This notebook helps you **evaluate RAG context** using Llumo’s powerful context-level metrics to ensure quality and safety:

✨ **Metrics included:**  
- **Context Utilization** ⚖️  
- **Redundancy Reduction** 🔍  
- **Relevance Retention** ☣️  
- **Semantic Cohesion**
- **Hallucination** 🚫  

---

### 🚀 **What you will do in this notebook:**  
1. 📂 Load input data from an Excel file  
2. 🤖 Evaluate the context for context utilization, relevance retention, hallucination etc.
3. 📊 View the detailed evaluation results  

---

🔐 **Note:** The Llumo API key will be securely requested during runtime using Colab’s input prompt.


 ### **⚙️ Step 1: Install Dependencies**

In [22]:
!pip install llumo -q
!pip install -q langchain-openai -q


### **📂 Step 2: Import Required Libraries**

In [4]:
import pandas as pd
import getpass
import os
import requests

### **🔑 Setup OpenAI API Key & Llumo API key**

In [1]:
import os

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "Enter Your Open API Key"

# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = "Enter Your LLumo Key"

openai_key = os.getenv("OPENAI_API_KEY")
llumo_key = os.getenv("LLUMO_API_KEY")

### **🧾 Step 3: Load the Dataset - Optional**

In [12]:
# Make sure 'data.xlsx' is uploaded to your Colab environment
df = pd.read_excel("data.xlsx") # We have list of queries

# Preview the data
df.head()


### **List to store query results as dictionaries — `[{},{},{},{}]`**

This list collects the output of multiple queries run through the agent.  
Each query result is stored as a dictionary containing:

- `query`: The input question  
- `context`: The contextual data retrieved from the database or other sources to assist in answering the query  
- `output`: The LLM final response as plain text

The data used for evaluation will be in the following Example format:

```
[  
  {
    "query": "What is the capital of France?",
    "context": "France is a country in Europe. Its capital city is Paris.",
    "output": "The capital of France is Paris."
  },
  {
    "query": "Summarize the plot of 'Romeo and Juliet'.",
    "context": 'Romeo and Juliet' is a tragedy by William Shakespeare.It is about two lovers from feuding families.",
    "output": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families."
  }
]
```



In [13]:
queries = df["query"].to_list()

### **🔐 Step 4: Getting the context from Database**

In [14]:
import requests


# Function to get context from external API
def get_context(query):
    url = "https://model-api.llumo.ai/functionCalling/get-context-from-db"
    reqBody = {"query": query}
    response = requests.post(url, json=reqBody)
    return response.json()["contexts"]

# Get all contexts in a single API call
contexts = get_context(queries)

In [15]:
# Prepare list of dicts [{query, context, output}, ...]
results = []
for query, context in zip(queries, contexts):
    results.append({"query": query, "context": context, "output": ""})


### **🧾 Step 5: Generating the Outputs**


In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Initialize LangChain OpenAI client
client = ChatOpenAI(
    model="gpt-4",
    temperature=0.7,
    api_key=openai_key  # Use your variable here
)

for item in results:
    query = item["query"]
    context = item["context"]


    # Prompt exactly like your original
    user_prompt = f"Give answer to the given query: {query}, using the given context: {context}."

    # Use LangChain to invoke the model
    response = client.invoke([HumanMessage(content=user_prompt)])

    # Save result in the same structure
    item["output"] = response.content

#  Now 'results' is your final [{},{},{}] list with 'query', 'context', 'output'

### 📄 **Input Data with keys — "query", "context", "output"**
Preview the enriched data that will be passed for output evaluation.


In [17]:
results[0]

{'query': 'How can I return a laptop if I am not satisfied with it?',
 'context': 'ElectraTech is your go-to destination for the latest in technology and electronics. Our offerings include the newest smartphones, laptops, smart home devices, and more. Take advantage of free shipping on orders over $150. All products come with a one-year warranty covering manufacturing defects. Returns are accepted within 30 days, provided the item is in its original, unopened packaging. Our dedicated customer support team is available 24\\/7 to help with any questions or concerns related to products, returns, or warranty claims. At ElectraTech, we are committed to providing exceptional products and outstanding service.',
 'output': 'If you are not satisfied with your laptop from ElectraTech, you can return it within 30 days of purchase. However, the laptop must be in its original, unopened packaging. If you have any questions or concerns regarding the return process, you can reach out to the dedicated 

### 🤖 **Step 5: Initialize Llumo Client And Evaluate Output**
This block initializes the `LlumoClient` and evaluates the quality and safety of output using selected KPIs like:

- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness

Additional Metrics:
- Context Utilization
- Hallucination
  


In [18]:

# Import the evaluation client from Llumo SDK
from llumo import LlumoClient

# Initialize the LlumoClient with your API key
client = LlumoClient(api_key = llumo_key)  # Replace with actual API key

resultDf = client.evaluateMultiple(
    data = results,  # Input Data
    evals = ["Context Utilization", "Redundancy Reduction", "Relevance Retention","Semantic Cohesion","Hallucination"], # List of metrics
    prompt_template = "Give answer to the given query: {{query}}, using the given context: {{context}}.",  # Prompt used for generation
    createExperiment = False,   # Set to True to save results as an experiment on the Llumo platform. If False, returns results as a DataFrame or A Python Dict. - Optional
    getDataFrame = True, # Return result as a DataFrame (True) or dictionary (False) - Optional
    )



Processing Batches: 100%|██████████| 5/5 [00:17<00:00,  3.50s/batch]


In [19]:
resultDf

,query,context,output,Context Utilization,Context Utilization Reason,Redundancy Reduction,Redundancy Reduction Reason,Relevance Retention,Relevance Retention Reason,Semantic Cohesion,Semantic Cohesion Reason,Hallucination,Hallucination Reason
0,How can I return a laptop if I am not satisfie...,ElectraTech is your go-to destination for the ...,If you are not satisfied with your laptop from...,99,The response accurately reflects the return po...,72,The context has some redundancy. The warranty...,100,The context provides all necessary information...,100,"The context presents information in a logical,...",13,The output paraphrases the context but provide...
1,What do I do if my product arrives with a defect?,FutureGadgets is your source for the most adva...,"If your product arrives with a defect, you can...",100,The response accurately reflects all relevant ...,79,The context has some redundancy. The warranty...,84,The context provides information about returns...,100,The context presents information in a logical ...,18,The output mentions a one-year warranty and cu...
2,How does CyberShield handle a data breach?,CyberShield Solutions is a premier provider of...,The text does not provide specific information...,100,The response accurately identifies the absence...,81,The context contains some redundancy. Phrases...,93,The context provides a good overview of CyberS...,99,The context presents a clear and coherent desc...,1,The output accurately reflects the context's l...
